In [1]:
import json
import sys
import pandas as pd

sys.path.append("../src")
import metrics
import sec_extract


def search_period_tags(cik, fiscal_year, keywords):
    """List full-year 10-K tags (income or cash flow) whose names contain any keyword, in $000s."""
    path = f"../data/raw/companyfacts/CIK{int(cik):010d}.json"
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    found = set()
    for tag, info in data["facts"].get("us-gaap", {}).items():
        if any(k in tag for k in keywords) and "USD" in info["units"]:
            for e in info["units"]["USD"]:
                if e.get("form") == "10-K" and "start" in e:          # has a start date = a period
                    start, end = pd.Timestamp(e["start"]), pd.Timestamp(e["end"])
                    full_year = 350 <= (end - start).days <= 380
                    same_year = (end - pd.Timedelta(days=15)).year == fiscal_year
                    if full_year and same_year:
                        found.add((tag, e["val"] / 1000))
    return sorted(found)


all_metrics = pd.read_csv("../data/processed/metrics.csv")
qnst_cik = all_metrics.loc[all_metrics["ticker"] == "QNST", "cik"].iloc[0]

for tag, value in search_period_tags(qnst_cik, 2026, ["PaymentsToAcquire", "BusinessCombinationConsideration"]):
    print(f"{tag:80} {value:>12,.0f}")

PaymentsToAcquirePropertyPlantAndEquipment                                              3,397


In [2]:
for tag, value in search_period_tags(qnst_cik, 2026, ["Acqui", "Business"]):
    print(f"{tag:80} {value:>12,.0f}")

BusinessCombinationContingentConsiderationArrangementsChangeInAmountOfContingentConsiderationLiability1        4,650
GoodwillAcquiredDuringPeriod                                                          136,742
NoncashOrPartNoncashAcquisitionValueOfLiabilitiesAssumed1                                   0
PaymentsToAcquirePropertyPlantAndEquipment                                              3,397
TreasuryStockValueAcquiredCostMethod                                                   31,442
